In [ ]:
# 1-2. Business context and data loading
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
orders, items, products = [pd.read_csv(f) for f in ["orders.csv", "order_items.csv", "products.csv"]]
for t in [orders, items, products]:
    print(t.shape, t.isna().sum()[lambda s: s > 0].to_dict())

In [ ]:
# 3. Data quality assessment
print("Duplicate order rows:", orders.duplicated().sum(), "| repeated IDs:", orders["order_id"].duplicated().sum())
print(orders["channel"].unique(), products["category"].unique(), orders["status"].value_counts().to_dict())
items.describe().round(2)

In [ ]:
# 4. Anomaly investigation
other = orders.loc[~orders["order_date"].str.match(r"\d{4}-"), "order_date"]
print(len(other), other.head(3).tolist(), "| day > 12:", (other.str[:2].astype(int) > 12).sum(), "| month > 12:", (other.str[3:5].astype(int) > 12).sum())

In [ ]:
li = items.merge(products[["product_id", "list_price"]], on="product_id", how="left")
li["price_ratio"] = li["unit_price"] / li["list_price"]
li["implied_qty"] = li["line_total"] / li["unit_price"]
li[(li["price_ratio"] > 2) | (li["quantity"] >= 50) | li["unit_price"].isna()]

In [ ]:
# 5. Transformation and reconciliation
o = orders.drop_duplicates().copy()
d = o["order_date"]
o["order_date"] = pd.to_datetime(d, format="ISO8601", errors="coerce").fillna(pd.to_datetime(d, format="%d/%m/%Y", errors="coerce"))
o["channel"] = o["channel"].str.strip().str.title()
o["customer_id"] = o["customer_id"].fillna("GUEST")
p = products.assign(category=products["category"].str.strip().str.title())
print(o["order_date"].isna().sum(), o["channel"].unique(), p["category"].unique())

In [ ]:
li["unit_price_raw"], li["quantity_raw"] = li["unit_price"], li["quantity"]
li["fix"] = np.select([li["price_ratio"] > 50, li["unit_price"].isna(), li["quantity"] == 999],
                      ["price in cents", "price from line total", "placeholder quantity"], "")
li.loc[li["fix"] == "price in cents", "unit_price"] /= 100
li["unit_price"] = li["unit_price"].fillna(li["line_total"] / li["quantity"])
li.loc[li["fix"] == "placeholder quantity", "quantity"] = li["implied_qty"]
li["fix"].value_counts()

In [ ]:
clean = (li.drop(columns=["list_price", "price_ratio", "implied_qty"])
           .merge(o, on="order_id", how="left", validate="m:1")
           .merge(p[["product_id", "product_name", "category"]], on="product_id", how="left", validate="m:1"))
clean["category"] = clean["category"].fillna("Unknown")
clean["exception"] = np.select([clean["status"].isna(), clean["category"] == "Unknown",
                                (clean["quantity"] * clean["unit_price"] - clean["line_total"]).abs() > 0.01],
                               ["order not found", "product not in catalog", "line total mismatch"], "")
clean["in_revenue"] = clean["status"] == "completed"
clean["exception"].value_counts()

In [ ]:
clean[clean["exception"] == "line total mismatch"].iloc[:, :6]

In [ ]:
# 6. Business analysis
rev = clean[clean["in_revenue"]]
order_val = rev.groupby(["order_id", "channel"], as_index=False)["line_total"].sum()
print("Revenue:", rev["line_total"].sum().round(2), "| Orders:", len(order_val), "| AOV:", order_val["line_total"].mean().round(2))
rev.groupby("category").agg(revenue=("line_total", "sum"), units=("quantity", "sum")).sort_values("revenue", ascending=False)

In [ ]:
rev.groupby("product_name")["line_total"].sum().nlargest(5)

In [ ]:
monthly = rev.groupby(rev["order_date"].dt.to_period("M"))["line_total"].sum()
pd.DataFrame({"revenue": monthly, "change": monthly.pct_change()}).round(2)

In [ ]:
rev[rev["order_date"].dt.month == 6].groupby("channel")["line_total"].agg(["sum", "count"])

In [ ]:
log_value = np.log10(order_val["line_total"])
q1, q3 = log_value.quantile([0.25, 0.75])
order_val["outlier"] = log_value > q3 + 1.5 * (q3 - q1)
print("Upper limit:", round(10 ** (q3 + 1.5 * (q3 - q1)), 2))
order_val.groupby("channel")["outlier"].agg(["sum", "count"])

In [ ]:
# 7. Visualizations
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
monthly.plot(ax=ax[0], marker="o", title="Monthly revenue ($)")
sns.boxplot(data=order_val, x="channel", y="line_total", log_scale=True, ax=ax[1]).set_title("Order value by channel ($)")
plt.show()

In [ ]:
# 8. Validation
print("Naive join:", items.merge(orders, on="order_id")["line_total"].sum().round(2), "| raw lines:", items["line_total"].sum().round(2))
print("Rows in:", len(items), "| rows out:", len(clean), "| category total:", rev.groupby("category")["line_total"].sum().sum().round(2))
clean.groupby(clean["status"].fillna("order not found"))["line_total"].sum().round(2)

In [ ]:
# 9. Deliverables
clean.to_csv("clean_sales.csv", index=False)
report = clean[(clean["fix"] != "") | (clean["exception"] != "") | ~clean["in_revenue"]]
report.to_csv("exceptions_report.csv", index=False)
print(len(clean), "clean lines |", len(report), "lines in the exception report")

## 9. Conclusions and recommendations
- **Data:** 20 duplicate orders removed (a join would have added $4,911.90), 127 day-first dates converted, and 26 order lines corrected with evidence (12 prices in cents, 10 missing prices, 4 placeholder quantities). Unresolved: 6 line-total mismatches, 8 lines for the unknown product P025, and 5 lines without an order ($395).
- **Business:** completed revenue is $433,779.55 from 1,328 orders (AOV $326.64). Bedding leads with $112,125.95. June rose 50% because of three large B2B orders, and November is the seasonal peak (+45%).
- **Outliers:** 36 high-value orders, all B2B. They are a different customer segment, not errors.
- **Recommendations:** enforce one date format and prices in dollars in the export, block the 999 quantity, add P025 to the catalog, and report B2B separately.
- **Final notes:** the three input files are synthetic and were generated by Gemini before recording.